## Project: Grover’s Algorithm

**Instructor:** Hoa Nguyen, CSIRO's Data61

### Project overview

In this project, you will implement and analyse **Grover's quantum search algorithm** for unstructured search in **n-qubit** systems. You will start with the classical case of searching for a **single marked item**, then extend your implementation to handle **k > 1 marked items**.

### Deliverables

1. Multiple **oracle** functions for different marked items
2. Working implementation of the **n-qubit** single-item Grover's search
3. An extended circuit implementation supporting **k = 2, 3, 4 marked items**
4. Analysis of the **optimal iteration count vs. k relationship**
5. Analysis showing how the Grover operator **amplifies target amplitudes**
6. Testing and validation on ideal and noisy quantum simulators

<div style="font-family: 'Arial'; font-size: 16px; line-height: 1.6; text-align: justify;">  

Each **Deliverable** below includes a task and a **sample solution**. Work through them in order. All circuits run on **simulators** (ideal and noisy). **Grover’s Search Algorithm** using Qiskit 2.x — (see deliverables above).

</div>

### **A friendly reminder**

<div style="font-family: 'Arial'; font-size: 16px; line-height: 1.6; text-align: justify;">   

Code snippets will be provided for you, except in certain cells where your task will be to complete the code to ensure our quantum program runs smoothly. These cells will be clearly marked with a comment like: <span style="font-family: monospace; font-weight: bold; color: #111; background-color: #fff8dc; padding: 2px 6px; border-radius: 4px;"> ### WRITE YOUR CODE BELOW THIS CELL ### </span>.

Another friendly reminder: we’ll be running our quantum programs exclusively on simulators. No worries—these work perfectly well for our purposes.
</div>

## **Pre-checking and Imports**


<div style="font-family: 'Arial'; font-size: 16px; line-height: 1.6; text-align: justify;">   

You should check your Qiskit version before starting the lab. It is recommended to use Qiskit version 2.0 or higher for the best experience.
</div>

In [ ]:
# If you run this notebook on Google Colab, please uncomment the following line and run it first to install the required packages
# !pip install -r https://raw.githubusercontent.com/hoaiocom/rmit-qss-2026/main/requirements.txt

# If you run on local machine, please follow the instructions in the README.md file to setup the virtual environment and install the required packages

In [ ]:
import qiskit
print(f"Qiskit version: {qiskit.__version__}")

In [ ]:
from qiskit_aer import AerSimulator
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile
from qiskit.visualization import plot_histogram

## **Introduction**

<div style="font-family: 'Arial'; font-size: 16px; line-height: 1.6; text-align: justify;">   
  
Grover's algorithm is a quantum algorithm designed to address the problem of searching for a marked item in an unsorted database, also referred to as an unstructured database. It was introduced by Lov Grover in 1996 and has been proven to provide a quantum advantage over classical computation.

To illustrate the notion of an unstructured database, consider a simple example with a list: `[2, 5, 3, 6, 8, 11, 9, 1]`. In the classical approach, the number of queries required depends on the position of the marked element. On average, one would need to check $\frac N 2$ elements, and in the worst case, $N$ elements, where $N$ denotes the size of the database. The corresponding time complexity is $O(N)$.

In contrast, the quantum approach using Grover’s algorithm achieves a time complexity of $O(\sqrt N)$ for the same problem. This improvement arises from the iterative process of applying the oracle, which inverts the amplitude of the target state, followed by amplitude amplification, thereby increasing the probability of measuring the correct result.
</div>

### **Main Procedure of Grover’s Algorithm**

<div style="font-family: 'Arial'; font-size: 16px; line-height: 1.6; text-align: justify;">   
  
The central idea of Grover’s algorithm can be summarized as follows:

- Prepare a quantum superposition over all possible states $|\psi \rangle = \frac{1}{\sqrt{N}} \sum_{x=0}^{N-1} |i\rangle_{basic}$, where $N=2^n$ represents the total number of basis states.


- Mark the target state, i.e., the state to be searched, by applying the oracle.

- Amplify the amplitude of the marked state.

- Repeat the above two steps n times (oracle application and amplitude amplification).

- Extract the classical result from the quantum circuit.

*Within the scope of this lab, we perform the search for one or more marked elements in a complete database of size $2^n$, where $n$ denotes the number of bits. This implies that, for  $n=5$, the search is carried out for the marked elements within the range $[0,2^5−1]$.*
</div>

## **Algorithm Implementation**

<div style="font-family: 'Arial'; font-size: 16px; line-height: 1.6; text-align: justify;">   

The figure below illustrates the design of Grover’s algorithm using a three-qubit quantum circuit.

<img src="../img/grover_circuit.png" alt="VS Code install prompt example" style="display: block; margin-left: auto; margin-right: auto; width: 74%; border: 1px solid #ccc;border-radius: 8px;">
</div>

In [ ]:
num_qubits = 3
qreg = QuantumRegister(num_qubits, name = "q")
creg = ClassicalRegister(num_qubits, name = "result")

# Marking the State
marked_states = ["010"]

# Since the order of bit strings in Qiskit is reversed, we invert the bit strings prior to marking them in the algorithm for visualization purposes.
reversed_states = [s[::-1] for s in marked_states]

### **Quantum Superposition Initialization**

In [ ]:
grover_initial = QuantumCircuit(qreg)
grover_initial.h(qreg)

grover_initial.draw('mpl')

## **Deliverable 1: Multiple oracle functions**

### **Oracle Design**

<div style="font-family: 'Arial'; font-size: 16px; line-height: 1.6; text-align: justify;">   

After preparing the quantum circuit in a superposition of all basis states, the next step is to design an operator, known as the Oracle, to mark the state that encodes the solution to the search problem.

The central idea of this step is to invert the amplitude of the marked states. Formally, this can be expressed as follows:
<div style="text-align: center;"> 

$O_w |\psi\rangle =
\begin{cases}
|\psi\rangle & \text{(} \psi \text{ is not the marked state)} \\
-|\psi\rangle & \text{(} \psi \text{ is the marked state)}
\end{cases}, \space \space \space$      *Let $O_w$ be the oracle, which will be constructed in the following section.*
</div>

For ease of understanding, consider the following example: let $n=3$, which implies that we have a total of 8 states. After preparing the quantum superposition state and marking the state $|100⟩$, the process can be illustrated as follows: 

<div style="text-align: center;"> 

$O_{w}|\psi\rangle = O_{w} \frac{1}{\sqrt{8}}\big( |000\rangle + |001\rangle + |010\rangle + |011\rangle + |100\rangle + |101\rangle + |110\rangle + |111\rangle \big)= \frac{1}{\sqrt{8}}\big(|000\rangle + |001\rangle + |010\rangle + |011\rangle - |100\rangle + |101\rangle + |110\rangle + |111\rangle \big)$
</div>
<div style="text-align: center;"> 

*Or*
</div>

<div style="text-align: center;"> 

$|\psi\rangle = \frac{1}{\sqrt{8}}
\begin{pmatrix}
1 \\
1 \\
1 \\
1 \\
1 \\
1 \\
1 \\
1
\end{pmatrix} \space \rightarrow \space O_w|\psi\rangle = \frac{1}{\sqrt{8}}
\begin{pmatrix}
1 \\
1 \\
1 \\
1 \\
-1 \\
1 \\
1 \\
1
\end{pmatrix}$
</div>

</div>

<div style="font-family: 'Arial'; font-size: 16px; line-height: 1.6; text-align: justify; color: #111; background-color: #fff8dc; padding: 15px; border-radius: 8px;">   

**Task:** Design of the Oracle
  
Your task is to **design an oracle** that implements **the amplitude inversion of one or more marked states**. The oracle can be constructed by the following procedure:

1. **Iterate** over all marked elements in the list.

2. For each marked bit string, apply **Pauli-X gates** to those qubits whose corresponding bit equals $0$.

3. Apply **a Hadamard gate** to the last qubit (the designated target qubit).

4. Apply **a multi-controlled $Z$ gate** (or equivalently a multi-controlled phase flip) with the last qubit as the target and the remaining qubits as controls.

5. Apply **a Hadamard gate** to the last qubit.

6. For each marked bit string, **undo** the initial Pauli-X gates by applying Pauli-X again to the qubits that were flipped in step $2$ (i.e., uncompute the bit flips).
</div>

**Sample solution:** Below.

In [ ]:
grover_oracle = QuantumCircuit(qreg)
grover_oracle.barrier()

### WRITE YOUR CODE BELOW THIS CELL ###

    
### YOUR CODE FINISHES HERE ###
grover_oracle.draw('mpl')

## **Grover Diffusion Operator Design**

<div style="font-family: 'Arial'; font-size: 16px; line-height: 1.6; text-align: justify;">  

After applying the Oracle, our quantum circuit undergoes a transformation in which the marked states acquire negative amplitudes compared to the others. However, the probability distribution across all states remains uniform, meaning that the marked states are still indistinguishable from the unmarked ones upon measurement.

To resolve this, it is necessary to design an operator that amplifies the amplitudes of the marked states. Mathematically, this operator—known as the diffusion or amplitude amplification operator—is defined as: 

<div style="text-align: center;"> 

$U_{D} = 2|s\rangle\langle s| - \mathbb{I}$
</div>

Here, the state $\ket s$ represents our quantum superposition state, while $U_D$ denotes the amplitude amplification operator, which we are currently designing. From a geometric perspective, this operator reflects the entire state vector about the axis $\ket s$.
</div>

<div style="font-family: 'Arial'; font-size: 16px; line-height: 1.6; text-align: justify; color: #111; background-color: #fff8dc; padding: 15px; border-radius: 8px;">   

**Exercise 2: Construction of the Grover Diffusion Operator**

Design the Grover amplification operator, which modifies the probability distribution over all basis states and increases the measurement probability of the marked states. The operator is implemented by the following sequence of operations:

1. Apply **a Hadamard gate** to every qubit.

2. Apply **Pauli-X gates** to every qubit.

3. Apply **a Hadamard gate** to the last qubit.

4. Apply **a multi-controlled $Z$ gate** with the last qubit as the target and the remaining qubits as controls.

5. Apply **a Hadamard gate** to the last qubit once more.

6. Apply **Pauli-X gates** to every qubit.

7. Apply **a Hadamard gate** to every qubit.
</div>

In [ ]:
grover_diffuser = QuantumCircuit(qreg)
grover_diffuser.barrier()

### WRITE YOUR CODE BELOW THIS CELL ###


### YOUR CODE FINISHES HERE ###

grover_diffuser.draw('mpl')

### **Optimal number of iterations**
<div style="font-family: 'Arial'; font-size: 16px; line-height: 1.6; text-align: justify;">   

In Grover’s algorithm, it is necessary to repeatedly apply the two operators we have just designed in order for the algorithm to correctly converge to the solution. The number of iterations for applying these two operators is referenced from <a href="https://quantum.cloud.ibm.com/learning/en/courses/fundamentals-of-quantum-algorithms/grover-algorithm/number-of-iterations?utm_source=chatgpt.com" target="_blank" style="color: #1d73e4ff; text-decoration: none; font-weight: bold;">IBM: Choosing the number of iterations</a>, and is computed according to the following formula:

<div style="text-align: center;"> 

$t_{\text{opt}} = \left\lfloor \frac{\pi}{4 \, \arcsin\!\left(\sqrt{\tfrac{M}{N}}\right)} \right\rfloor$
</div>

Here, $N$ denotes the total number of states corresponding to all possible qubit strings, while $M$ represents the number of marked states.
</div>

In [ ]:
import math

# Total number of states corresponding to the number of qubit strings
N = (2 ** num_qubits)
# Number of marked states
M = len(marked_states)
# Compute θ – used to determine the required number of iterations for the algorithm
theta = math.asin(math.sqrt(M / N))
# Compute the optimal number of iterations
t_opt = math.floor(math.pi / (4 * theta))

print("Optimal Number of Iterations:", t_opt)

## **Deliverable 2: Working n-qubit single-item Grover's search**

### **Design of the Complete Algorithm**

<div style="font-family: 'Arial'; font-size: 16px; line-height: 1.6; text-align: justify; color: #111; background-color: #fff8dc; padding: 15px; border-radius: 8px;">   

**Task:** Designing the Algorithm
  
The task is to assemble all components to construct the complete algorithm, which proceeds step by step as follows:

1. Integrate **grover_initial**.

2. Integrate **both grover_oracle** and **grover_diffuser**.

3. **Repeat** the above integration for **$t_{opt}$ iterations**.

4. Apply **measurement gates** to all quantum registers and record the outcomes in the classical register.
</div>

**Sample solution:** Below.

In [ ]:
grover_algorithm = QuantumCircuit(qreg, creg)

### WRITE YOUR CODE BELOW THIS CELL ###


### YOUR CODE FINISHES HERE ###

grover_algorithm.draw('mpl')

In [ ]:
shots = 1024
backend_simulator = AerSimulator()

transpiled_grover_circuit = transpile(grover_algorithm, backend_simulator)
job = backend_simulator.run(transpiled_grover_circuit, shots = shots)
results = job.result()
counts = results.get_counts()

plot_histogram(counts)

<div style="font-family: 'Arial'; font-size: 16px; line-height: 1.6; text-align: justify;">   
  
✨ Ta-da! As the results indicate, it is evident that after executing the algorithm, all the initial quantum superposition states have converged to the marked states, demonstrating that our algorithm performs effectively.

🚀 **A little secret**: you can go back to the very first steps of the algorithm, where the number of qubits and the marked states are defined, and modify them. For example, try increasing the number of qubits or marking more than one state instead of just one. You might be surprised by what you discover!
</div>

## **Deliverable 3: Extended circuit for k = 2, 3, 4 marked items**

<div style="font-family: 'Arial'; font-size: 16px; line-height: 1.6; text-align: justify; color: #111; background-color: #fff8dc; padding: 15px; border-radius: 8px;">

**Task:** Using the same oracle and diffuser design, build and run the Grover circuit for **k = 2**, **k = 3**, and **k = 4** marked items (e.g. on 3 or 4 qubits). For each case, set `marked_states` to a list of 2, 3, or 4 distinct basis-state strings, recompute `t_opt`, rebuild the circuit, run it, and show the measurement histogram so that the marked states are clearly favoured.
</div>

In [ ]:
### WRITE YOUR CODE BELOW THIS CELL ###


### YOUR CODE FINISHES HERE ###

## **Deliverable 4: Optimal iteration count vs k analysis**

<div style="font-family: 'Arial'; font-size: 16px; line-height: 1.6; text-align: justify; color: #111; background-color: #fff8dc; padding: 15px; border-radius: 8px;">

**Task:** Analyse how the **optimal number of Grover iterations** $t_{opt} = \lfloor \pi/(4\theta) \rfloor$ (with $\theta = \arcsin\sqrt{M/N}$, $M$ = number of marked items, $N = 2^n$) depends on **k** (and optionally on **n**). Produce a table or plot showing $t_{opt}$ vs k for at least one fixed n, and briefly comment on the relationship.
</div>

In [ ]:
### WRITE YOUR CODE BELOW THIS CELL ###


### YOUR CODE FINISHES HERE ###

## **Deliverable 5: Amplitude amplification analysis**

<div style="font-family: 'Arial'; font-size: 16px; line-height: 1.6; text-align: justify; color: #111; background-color: #fff8dc; padding: 15px; border-radius: 8px;">

**Task:** Show how the Grover operator **amplifies the target (marked) amplitudes** over iterations. For a small example (e.g. n = 3, one marked state), run the circuit with 0, 1, 2, … iterations (or use the statevector) and plot the **probability of measuring the marked state(s)** vs iteration number, so that amplification is clearly visible.
</div>

In [ ]:
### WRITE YOUR CODE BELOW THIS CELL ###


### YOUR CODE FINISHES HERE ###

## **Deliverable 6: Testing on ideal and noisy simulators**

<div style="font-family: 'Arial'; font-size: 16px; line-height: 1.6; text-align: justify; color: #111; background-color: #fff8dc; padding: 15px; border-radius: 8px;">

**Task:** Run the same Grover circuit on an **ideal simulator** and on a **noisy simulator** (e.g. Qiskit Aer with a noise model, or a fake backend). Compare the measurement histograms and briefly comment on how noise affects the success probability of finding the marked state(s).
</div>

In [ ]:
### WRITE YOUR CODE BELOW THIS CELL ###


### YOUR CODE FINISHES HERE ###

<div style="font-family: 'Arial'; font-size: 16px; line-height: 1.6; text-align: justify;">   

🎉 Congratulations! We have successfully completed the Grover's algorithm implementation

</div>